# Predictive E-Commerce Behavioral Analytics
## Colab execution notebook
All real-data preparation, model fitting, evaluation and CSV generation happen in this Colab runtime. Power BI Desktop only imports the resulting tables.

**Before running:** Use a standard Python 3.12+ CPU runtime. A GPU is not needed. Execute cells in order. Download this repository ZIP and this notebook, or open the notebook from your GitHub repository. No credentials belong in this notebook.

This project uses UCI Online Retail (2010–2011), not contemporary 2026 transactions. Performance is generated by your run; no scores are prefilled. Review `docs/METHODOLOGY.md` before interpreting results.

In [ ]:
import sys, os, subprocess, pathlib, re, zipfile, io
try:
    from google.colab import files, drive
except ImportError as exc:
    raise RuntimeError("Run this notebook in Google Colab, not locally.") from exc
assert sys.version_info >= (3, 12), "Choose a Python 3.12+ Colab runtime."
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'

# Fill in your PUBLIC repository URL after creating it. Leave blank to upload the ZIP.
REPO_URL = "https://github.com/ashyx12/minor-spec"
REPO_DIR = pathlib.Path('/content/ecommerce-analytics')
if not (REPO_DIR / 'pyproject.toml').exists():
    if REPO_URL:
        if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+', REPO_URL.rstrip('/')):
            raise ValueError("Use a public https://github.com/owner/repository URL, without a token.")
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        print('Upload ecommerce-analytics.zip (the complete repository bundle).')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload exactly one repository ZIP.')
        with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
            # Extract only repository content and reject unsafe archive paths.
            names = archive.namelist()
            roots = [n[:-len('pyproject.toml')] for n in names if n.endswith('pyproject.toml')]
            if len(roots) != 1:
                raise ValueError('Expected one project root.')
            prefix = roots[0]
            for member in archive.infolist():
                if not member.filename.startswith(prefix) or member.is_dir():
                    continue
                rel = pathlib.PurePosixPath(member.filename[len(prefix):])
                if rel.is_absolute() or '..' in rel.parts:
                    raise ValueError('Unsafe ZIP member.')
                target = REPO_DIR.joinpath(*rel.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)

## Install reproducible dependencies
If Colab asks for a runtime restart after installation, restart it, rerun the setup cell above, then continue below. The repository remains in `/content` across a runtime restart (not a disconnect/delete).

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]'], check=True)
# Pipeline execution uses a fresh subprocess so it sees the newly installed packages.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_core.py'], check=True)

## Save runs to Google Drive
Drive is optional but recommended because Colab runtimes expire. Each run gets its own directory. Stage outputs are written there as they complete; a failed run keeps its error manifest. The runner does not silently resume partial runs—start a new run after fixing the error.

In [ ]:
from datetime import datetime, timezone
USE_DRIVE = True
if USE_DRIVE:
    drive.mount('/content/drive')
    RUN_ROOT = pathlib.Path('/content/drive/MyDrive/ecommerce-analytics/runs')
else:
    RUN_ROOT = pathlib.Path('/content/retail-runs')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT = RUN_ROOT / RUN_ID
print('Run outputs:', OUT)

## Acquire the original dataset
Downloads from UCI without Kaggle authentication. If unavailable, download the original `Online Retail.xlsx` from [UCI](https://archive.ics.uci.edu/dataset/352/online+retail), set `UPLOAD_WORKBOOK=True`, and upload it here. Do not substitute a different dataset without adapting the date cutoffs and data contract.

In [ ]:
UPLOAD_WORKBOOK = False
DATA = pathlib.Path('/content/retail-data/Online Retail.xlsx')
DATA.parent.mkdir(exist_ok=True)
if UPLOAD_WORKBOOK:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload one original workbook.')
    DATA.write_bytes(next(iter(uploaded.values())))
else:
    subprocess.run([sys.executable, '-c',
        'from retail_analytics.data import download; import sys; download(sys.argv[1])', str(DATA)], check=True)
print(DATA, DATA.stat().st_size, 'bytes')

## Run the complete experiment
The default dates match UCI Online Retail. Recommendation validation: September 2011; test: October 2011. Repeat-purchase model: July/August/September snapshots for training, October for validation, November for testing, each with a 30-day label window.

The miner bounds its search to 300 products and itemsets of up to 3 products. This is an explicit scope restriction for CPU/RAM practicality. Runtime depends on Colab resources; do not expect a guaranteed duration. Watch progress in the cell output.

In [ ]:
subprocess.run([sys.executable, '-m', 'retail_analytics.pipeline',
                '--input', str(DATA), '--output', str(OUT)], check=True)

## Inspect evidence before writing conclusions
Compare baselines even if the hybrid model loses. Confidence intervals here are customer bootstrap summaries of a fixed trained model. They do not account for every source of uncertainty. Offline ranking does not demonstrate an increase in revenue.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
manifest = json.loads((OUT/'manifest.json').read_text())
assert manifest['status'] == 'complete'
display(pd.read_csv(OUT/'recommendation_test_metrics.csv'))
display(pd.read_csv(OUT/'prediction_test.csv'))
display(pd.read_csv(OUT/'cluster_diagnostics.csv'))
print(json.loads((OUT/'prediction_metadata.json').read_text()))
print(json.loads((OUT/'data_quality.json').read_text()))
for name in ['monthly_revenue', 'cluster_selection', 'segment_profiles',
             'recommendation_comparison', 'prediction_calibration', 'permutation_importance']:
    display(Image(filename=str(OUT/'figures'/f'{name}.png')))

## Query a customer or use the cold-start fallback
Only load model files produced by your own run: joblib files can execute code. A recommendation score is a ranking score, not a purchase probability. Previously purchased products remain eligible because this is a repeat-purchase task.

In [ ]:
# Use a fresh subprocess to avoid notebook package-version conflicts after installation.
CUSTOMER_ID = '17850'  # Change to an ID in DimCustomer, or an unknown ID to test fallback.
script = """
import joblib, json, sys
from pathlib import Path
out = Path(sys.argv[1])
model = joblib.load(out/'recommendation_production.joblib')
alpha = json.loads((out/'manifest.json').read_text())['selected_hybrid_alpha']
for item, score in model.recommend(sys.argv[2], k=10, alpha=alpha):
    print(item, round(score, 6))
"""
subprocess.run([sys.executable, '-c', script, str(OUT), CUSTOMER_ID], check=True)

## Export to Power BI Desktop
Download the run ZIP. Extract it on the machine running Power BI. Follow `powerbi/BUILD_GUIDE.md`, import the `powerbi` CSV folder, and add the supplied DAX measures. No model training or dataset preprocessing needs to run on your computer.

The generated CSVs and trained artifacts are intentionally excluded from Git. Keep run evidence on Drive; selectively publish aggregate metrics and figures after reviewing them.

In [ ]:
import shutil
zip_path = shutil.make_archive(f'/content/retail-results-{RUN_ID}', 'zip', OUT)
files.download(zip_path)